# SentiVerse AI - Developer Pipeline Walkthrough

This interactive notebook demonstrates each stage of the SentiVerse AI social media intelligence pipeline, from data collection to advanced classification models and business intelligence insights.

## Environment Setup & Imports

First, we import the core modules of the SentiVerse system. Ensure your working directory is set to the repository root so the `src` packages can be correctly imported.

In [ ]:
import os
import sys
from datetime import datetime

# Adjust system path to import src modules if running from notebooks directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data.collector import SocialDataCollector
from src.preprocessing.cleaner import TextCleaner
from src.preprocessing.nlp_pipeline import NLPPipeline
from src.features.text_stats import TextStatsExtractor
from src.features.vectorizers import TextVectorizer
from src.models.classical_ml import ClassicalMLWorkbench
from src.business.emotion_detector import EmotionDetector
from src.business.advanced_detectors import AdvancedDetectors
from src.business.bi_engine import BIEngine

## Stage 1: Data Collection & Simulation

We use the `SocialDataCollector` to generate realistic mock posts for simulation (simulating a Reddit feed for the query *SentiVerse App*).

In [ ]:
collector = SocialDataCollector()
raw_df = collector._generate_mock_posts(platform="reddit", count=10, query="SentiVerse App")
print(f"Generated {len(raw_df)} mock Reddit posts for simulation.")
print("Sample raw text:", raw_df['text'].iloc[0][:100] + "...")

## Stage 2: Data Cleaning & Reduction Summary

Next, we configure a cleaning pipeline that lowercases text, removes URLs, expands contractions, and strips punctuation. We'll clean the raw feed and output the reduction percentage.

In [ ]:
cleaner = TextCleaner({
    "lowercase": True,
    "remove_urls": True,
    "expand_contractions": True,
    "remove_punctuation": True
})

cleaned_texts = []
stats_list = []
for text in raw_df['text']:
    clean_text, stats = cleaner.clean_text(text)
    cleaned_texts.append(clean_text)
    stats_list.append(stats)
    
raw_df['cleaned_text'] = cleaned_texts
print("Sample cleaned text:", raw_df['cleaned_text'].iloc[0])

summary = TextCleaner.get_cleaning_pipeline_summary(raw_df['text'], raw_df['cleaned_text'])
print(f"Average length reduction: {summary['reduction_percentage']}% (from {summary['average_original_length']} to {summary['average_cleaned_length']} characters)")

## Stage 3: NLP Pipeline Analysis

We process the cleaned comments to extract linguistic features, tokenizing them, tagging parts-of-speech (POS), and detecting named entities (NER).

In [ ]:
nlp = NLPPipeline(use_spacy=False)
sample_nlp = nlp.process_pipeline(raw_df['cleaned_text'].iloc[0])
print("Tokens (lemmatized/stopwords removed):", sample_nlp['tokens'][:5], "...")
print("NER entities detected:", sample_nlp['entities'])

## Stage 4: Feature Engineering

We engineer numerical features from the raw texts like character counts, word counts, capital-to-lowercase ratios, emoji counts, and punctuation counts.

In [ ]:
stats_extractor = TextStatsExtractor()
feat_df = stats_extractor.extract_features_df(raw_df, text_column='text')
print("Engineered features:", [col for col in feat_df.columns if col not in raw_df.columns])
print("Sample character counts:", feat_df['char_count'].head(3).tolist())

## Stage 5: Model Training Workbench

We vectorize the corpus into a TF-IDF matrix representation and train a classical Logistic Regression model using cross-validation.

In [ ]:
vectorizer = TextVectorizer(method="tfidf", max_features=100)
X = vectorizer.fit_transform(raw_df['cleaned_text'].tolist())

# Assign binary labels for demonstration
y = [1 if idx % 2 == 0 else 0 for idx in range(len(raw_df))]

workbench = ClassicalMLWorkbench(random_state=42)
model, cv_score = workbench.train(
    model_name="Logistic Regression",
    X_train=X,
    y_train=y,
    cv=2
)
print(f"Logistic Regression Model trained. Mean CV Accuracy: {cv_score:.4f}")

## Stage 6: Emotion & Quality Attributes Filters

We run our lexicon-based Emotion Detector and the rule-based Quality Filters on a sample critical comment to analyze toxic, sarcastic, spam, and fake news indicators.

In [ ]:
emotion_detector = EmotionDetector()
advanced_detectors = AdvancedDetectors()

sample_text = "I absolutely hate this laggy app! It crashes my computer!"
emotion, _ = emotion_detector.predict_emotion(sample_text, sentiment_score=-0.8)
adv_flags = advanced_detectors.analyze_post(sample_text, sentiment_score=-0.8)

print(f"Text: '{sample_text}'")
print(f"Detected Emotion: {emotion.upper()}")
print(f"Toxic: {adv_flags['is_toxic']} | Sarcastic: {adv_flags['is_sarcastic']} | Spam: {adv_flags['is_spam']}")

## Stage 7: Business Intelligence & KPIs

Finally, we process the dataset through our BIEngine to extract KPIs like the Brand Reputation Score (with exponential time decay), Estimated CSAT, and generate actionable engineering and business suggestions.

In [ ]:
# Add mock sentiment classifications to df
raw_df['sentiment'] = ["Positive" if idx % 2 == 0 else "Negative" for idx in range(len(raw_df))]
raw_df['sentiment_score'] = [0.6 if idx % 2 == 0 else -0.5 for idx in range(len(raw_df))]

bi = BIEngine()
reputation = bi.calculate_brand_reputation_score(raw_df)
csat = bi.calculate_csat_score(raw_df)
suggestions = bi.generate_product_suggestions(raw_df)

print(f"Calculated Brand Reputation: {reputation}/100")
print(f"Estimated Customer Satisfaction (CSAT): {csat}%")
print("\nActionable Product / Engineering Suggestions:")
for sug in suggestions:
    print(f" - [{sug['category']}] {sug['action']} (Trigger: {sug['issue']})")